# 02 - position-weighted DFlash loss와 anchor 격리

**학습 목표**: 논문 식 (1)-(2)의 exponential weight와 여러 anchor block이 서로 보지 않는 visibility를 계산합니다.

**실행 방법**: Python 3/Jupyter에서 cell을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 라이브러리 `math`만 사용합니다.

In [ ]:
import math
# list로 위치별 weight를 보존하면 가까운 token과 먼 token의 감쇠를 직접 비교할 수 있습니다.

def position_weights(block_size=16, gamma=7.0):
    # anchor k=0은 제외하고 먼 future token일수록 작은 weight를 줍니다.
    return [math.exp(-max(k - 1, 0) / gamma) for k in range(1, block_size)]

def weighted_loss(negative_log_probs, gamma=7.0):
    weights = position_weights(len(negative_log_probs) + 1, gamma)
    return sum(w * loss for w, loss in zip(weights, negative_log_probs)) / sum(weights)

losses = [0.10 + 0.04 * k for k in range(15)]
weights = position_weights(16, gamma=7.0)
print('first/last weight:', round(weights[0], 3), round(weights[-1], 3))
print('weighted loss:', round(weighted_loss(losses), 4))

# target hidden-state 열 h0..h3, 이어서 두 draft block의 mask 열입니다.
columns = ['h0','h1','h2','h3','A0','A1','A2','B0','B1','B2']
anchors = {'A': 2, 'B': 4}
def visible(block, column):
    if column.startswith('h'):
        return int(column[1:]) < anchors[block]
    return column.startswith(block)

for block in ('A', 'B'):
    row = ''.join('#' if visible(block, col) else '.' for col in columns)
    print(block, row, columns)

assert weights[0] > weights[-1]
assert visible('A', 'A2') and not visible('A', 'B0')
assert visible('B', 'h3') and not visible('A', 'h3')


실제 구현은 K=16 anchor를 한 FlexAttention pass에 묶습니다. 여기서는 두 block만 그려, target prefix는 anchor 전까지 보되 draft block끼리는 격리되는 원리를 확인합니다.